# Lesson 3: Building an Agent Reasoning Loop

Build a multi-step research agent over a PDF using Gemini function calling and LlamaIndex's `FunctionAgent` workflow API.

## Setup

In [ ]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Load the data

Ensure `metagpt.pdf` is in this folder:

```bash
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

## Setup the Query Tools

In [ ]:
from utils import get_doc_tools

vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

## Setup Function Calling Agent

Modern LlamaIndex uses `FunctionAgent` (the older `FunctionCallingAgentWorker` / `AgentRunner` APIs were removed).

In [ ]:
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context
from helper import DEFAULT_LLM_MODEL

llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY, temperature=0)

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful research assistant over the MetaGPT paper. "
        "Always use the provided tools to answer questions."
    ),
    verbose=True,
)

# Context holds conversation memory across turns
ctx = Context(agent)

In [ ]:
response = await agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other.",
    ctx=ctx,
)
print(str(response))

In [ ]:
# Inspect tool call sources when available
if getattr(response, "tool_calls", None):
    print(response.tool_calls)
else:
    print(response)

In [ ]:
response = await agent.run(
    "Tell me about the evaluation datasets used.",
    ctx=ctx,
)
print(str(response))

In [ ]:
response = await agent.run(
    "Tell me the results over one of the above datasets.",
    ctx=ctx,
)
print(str(response))

## Lower-Level: Debuggability and Control

Instead of the old `create_task` / `run_step` API, stream workflow events to observe each tool call as it happens.

In [ ]:
from llama_index.core.agent.workflow import AgentStream, ToolCall, ToolCallResult

agent = FunctionAgent(
    tools=[vector_tool, summary_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful research assistant over the MetaGPT paper. "
        "Always use the provided tools to answer questions."
    ),
    verbose=True,
)
ctx = Context(agent)

In [ ]:
handler = agent.run(
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other.",
    ctx=ctx,
)

async for event in handler.stream_events():
    if isinstance(event, ToolCall):
        print(f"[tool call] {event.tool_name}({event.tool_kwargs})")
    elif isinstance(event, ToolCallResult):
        print(f"[tool result] {event.tool_name} -> {str(event.tool_output)[:300]}...")
    elif isinstance(event, AgentStream):
        # token stream (optional)
        pass

response = await handler
print("\nFinal response:\n", str(response))

In [ ]:
# Continue the same conversation with extra guidance mid-reasoning
handler = agent.run(
    "What about how agents share information?",
    ctx=ctx,
)
response = await handler
print(str(response))